# Zero-shot Quickstart (Multiple Samples)

This quickstart embeds **several** spatial samples together — different sections or donors — and shows that TERRA's embeddings integrate them into a shared space, so cell types and niches line up across samples.

The only differences from the {doc}`single-sample quickstart <zero_shot_quickstart>` are that we concatenate multiple samples into one `AnnData` and pass `sample_key` so each section is tokenized with its own spatial graph.

:::{note}
**TERRA requires an NVIDIA GPU** for the embedding step. This page ships with its outputs cleared — run it on a GPU machine to reproduce the figures.
:::

## 1. Setup

In [ ]:
import logging

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq

from terra import download_pretrained, harmonize_tokenize_embed_pipeline

logging.basicConfig(level="INFO")
sc.settings.set_figure_params(dpi=80, frameon=False)

In [ ]:
import os
import shutil
import urllib.request


def fetch_10x_xenium(base_url, out_dir):
    """Download a 10x Xenium sample (counts matrix + cell metadata) and load it
    as an AnnData with single-cell spatial coordinates in ``.obsm['spatial']``.

    Only the two small standalone files are fetched (a few tens of MB), not the
    multi-GB output bundle. Files are cached in ``out_dir`` on first run.
    """
    os.makedirs(out_dir, exist_ok=True)
    for fname in ["cell_feature_matrix.h5", "cells.parquet"]:
        dest = os.path.join(out_dir, fname)
        if not os.path.exists(dest):
            print(f"Downloading {fname} ...")
            # 10x rejects the default urllib User-Agent, so set a browser one.
            req = urllib.request.Request(f"{base_url}_{fname}",
                                         headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req) as r, open(dest, "wb") as f:
                shutil.copyfileobj(r, f)

    adata = sc.read_10x_h5(os.path.join(out_dir, "cell_feature_matrix.h5"))
    adata.var_names_make_unique()
    # Drop control probes / blank codewords; keep real gene-expression features.
    adata = adata[:, adata.var["feature_types"] == "Gene Expression"].copy()

    cells = pd.read_parquet(os.path.join(out_dir, "cells.parquet")).set_index("cell_id")
    adata.obs = adata.obs.join(cells)
    adata.obsm["spatial"] = adata.obs[["x_centroid", "y_centroid"]].to_numpy()
    return adata

## 2. Download multiple samples

We use the two serial-section replicates of the 10x Genomics **Xenium Human Breast Cancer** dataset as two samples (~10 MB each). Any two sections sharing the same gene panel work the same way — including two Xenium Prime samples for a larger panel.

In [ ]:
def load_sample(base_url, out_dir, sample):
    adata = fetch_10x_xenium(base_url, out_dir)
    # Namespace cell ids by sample so they stay unique after concatenation.
    adata.obs["sample"] = sample
    adata.obs_names = sample + "_" + adata.obs_names.astype(str)
    adata.obs["cell_id"] = adata.obs_names.astype(str)
    # Crop a contiguous window per section to keep the tutorial fast.
    xy = adata.obsm["spatial"]
    x0, y0 = xy.min(0)
    keep = (xy[:, 0] <= x0 + 750) & (xy[:, 1] <= y0 + 750)
    return adata[keep].copy()


base = "https://cf.10xgenomics.com/samples/xenium/1.0.1"
rep1 = load_sample(f"{base}/Xenium_FFPE_Human_Breast_Cancer_Rep1/Xenium_FFPE_Human_Breast_Cancer_Rep1",
                   "data/breast_rep1", "rep1")
rep2 = load_sample(f"{base}/Xenium_FFPE_Human_Breast_Cancer_Rep2/Xenium_FFPE_Human_Breast_Cancer_Rep2",
                   "data/breast_rep2", "rep2")

adata = ad.concat([rep1, rep2])
print(adata.obs["sample"].value_counts())
print(adata)

In [ ]:
# Total counts per cell, in space, for each section.
adata.obs["total_counts"] = np.asarray(adata.X.sum(axis=1)).ravel()
for sample in adata.obs["sample"].unique():
    sq.pl.spatial_scatter(adata[adata.obs["sample"] == sample],
                          color="total_counts", shape=None, size=4, title=sample)

## 3. Download a pretrained TERRA model

In [ ]:
model_dir = download_pretrained("Lotfollahi-lab/TERRA-96M")

## 4. Embed all samples together (zero-shot)

Pass `sample_key="sample"` so each section is tokenized with its own spatial neighborhood graph (neighborhoods never cross sections); `batch_key` labels the batch variable for downstream comparison.

In [ ]:
adata = harmonize_tokenize_embed_pipeline(
    adata=adata,
    sample_key="sample",      # tokenize each section separately
    batch_key="sample",
    model_folder_path=model_dir,
    cache_directory_path="./terra_cache",
)
print(list(adata.obsm))

## 5. Visualize — do the samples integrate?

Color the UMAP by `sample`. Well-integrated embeddings mix the sections (shared biology overlaps) rather than separating by batch.

In [ ]:
for emb_key in ["cell_emb", "neighborhood_emb"]:
    sc.pp.neighbors(adata, use_rep=emb_key, key_added=emb_key)
    sc.tl.umap(adata, neighbors_key=emb_key)
    sc.pl.umap(adata, neighbors_key=emb_key, color="sample", title=f"{emb_key} by sample")

## 6. Niche & cell-type identification across samples

Cluster once over the combined embedding; the same clusters now span both sections.

In [ ]:
sc.tl.leiden(adata, neighbors_key="neighborhood_emb", key_added="niche",
             resolution=0.6, flavor="igraph", n_iterations=2, directed=False)
sc.tl.leiden(adata, neighbors_key="cell_emb", key_added="cell_type",
             resolution=0.6, flavor="igraph", n_iterations=2, directed=False)

sc.tl.umap(adata, neighbors_key="neighborhood_emb")
sc.pl.umap(adata, neighbors_key="neighborhood_emb", color=["niche", "sample"])

In [ ]:
# The shared niches, shown in space for each section.
for sample in adata.obs["sample"].unique():
    sq.pl.spatial_scatter(adata[adata.obs["sample"] == sample],
                          color="niche", shape=None, size=6, title=sample)

## Next steps

Continue with {doc}`downstream_analysis` for gene-level embeddings, spatial gene-pair scoring, EMD spatial structure, and perturbation.